# Federal Reserve Macro Insights MVP (Streamlined)

This notebook is the clean orchestrator version.
Core logic is now in `core/*.py` modules.

Original notebook preserved: `fed_macro_mvp.ipynb`.


## 0) Install / refresh dependencies

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 1) Imports and configuration

In [ ]:
from pathlib import Path
import pandas as pd

from core.pipeline import create_config, run_ingest_and_index, run_full_analysis, persist_results

cfg = create_config(Path.cwd())
print('Project dir:', cfg.project_dir)
print('Model:', cfg.ollama_model)


/Users/atheeshkrishnan/AK/DEV/hawkdove/storied/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Optional runtime overrides

In [ ]:
# Recommended baseline on local CPU
cfg.top_k_topic = 2
cfg.max_context_chars = 2400
cfg.ollama_num_predict = 380
cfg.enable_reranker = False

# Keep retries + robust parser enabled
cfg.ollama_max_retries = 3
cfg.retry_context_shrink = [1.0, 0.85, 0.7]
cfg.retry_predict_shrink = [1.0, 1.0, 0.9]
cfg.use_json_repair = True

print('Overrides applied.')

## 3) Ingestion + indexing

In [ ]:
ingest_index_result = run_ingest_and_index(cfg)

catalog_df = ingest_index_result['catalog_df']
download_df = ingest_index_result['download_df']
chunks_df = ingest_index_result['chunks_df']

print('Catalog candidates:', len(catalog_df))
print('Downloaded/available:', len(download_df))
print('Chunks:', len(chunks_df))

if not download_df.empty:
    display(download_df[['title', 'status', 'local_path']].head(10))

## 4) Macro analysis

In [ ]:
analysis_result = run_full_analysis(cfg)

print('Sparse enabled:', analysis_result['sparse_enabled'])
print('Reranker enabled:', analysis_result['reranker_enabled'])
print('Timings:', analysis_result['timings'])

display(analysis_result['topic_summary_df'])
if not analysis_result['hits_df'].empty:
    display(analysis_result['hits_df'][['topic', 'chunk_id', 'doc_id', 'final_score', 'recency']].head(12))
if not analysis_result['attempt_log'].empty:
    display(analysis_result['attempt_log'])

print(analysis_result.get('normalized_json_text', analysis_result['llm_text']))
if analysis_result['parsed'] is None:
    print('[check] JSON parse failed after retries')
else:
    quality = analysis_result['quality']
    print('[check] Missing top keys:', quality['missing_top_keys'] if quality['missing_top_keys'] else 'None')
    print('[check] Bad shape:', quality['bad_shape'] if quality['bad_shape'] else 'None')
    print('[check] Bad evidence IDs:', quality['bad_evidence_ids'] if quality['bad_evidence_ids'] else 'None')
    print('[check] Unknown citation IDs:', quality['unknown_citation_ids'] if quality['unknown_citation_ids'] else 'None')
    print('[check] Missing topics:', quality['missing_topics'] if quality['missing_topics'] else 'None')
    print('[check] Extra topics:', quality['extra_topics'] if quality['extra_topics'] else 'None')

## 5) Save artifacts

In [ ]:
saved_paths = persist_results(cfg, analysis_result)
print(saved_paths)